# Can deeper reasoning catch incompatible gear and safety constraints before we recommend a camping bundle?

## 1. Before You Begin

The Contoso Outdoors catalog and manuals include details that deserve a second
look before we answer a shopper. Our single focus is **constraint reasoning**:
separate facts from conflicts and unknowns before making a recommendation.

Deploy `gpt-6-sol` in Microsoft Foundry, configure section 2, and review the
[repository quickstart](../../quickstart/README.md). Consult the
[Azure OpenAI pricing page](https://azure.microsoft.com/pricing/details/azure-openai/)
for current rates.


In [ ]:
# 2. Verify your environment
import os
from pathlib import Path

from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()


def find_assets() -> Path:
    """Find the shared data whether Jupyter starts here or at the repo root."""
    for base in (Path.cwd(), *Path.cwd().parents):
        candidate = base / "models/azure-openai/shared/contoso-outdoors"
        if (candidate / "products.json").is_file():
            return candidate
    raise FileNotFoundError(
        "Could not find models/azure-openai/shared/contoso-outdoors. "
        "Run this notebook from a checkout of the model-releases repository."
    )


required = ["AZURE_OPENAI_ENDPOINT", "AZURE_OPENAI_API_KEY", "AZURE_OPENAI_GPT_6_SOL_DEPLOYMENT"]
missing = [name for name in required if not os.getenv(name)]
if missing:
    raise EnvironmentError(
        f"Missing {missing}. Copy scripts/sample.env to .env, add the values, "
        "and review models/quickstart/README.md."
    )

endpoint = os.environ["AZURE_OPENAI_ENDPOINT"].rstrip("/")
base_url = endpoint if endpoint.endswith("/openai/v1") else f"{endpoint}/openai/v1"
client = OpenAI(api_key=os.environ["AZURE_OPENAI_API_KEY"], base_url=f"{base_url}/")
deployment = os.environ["AZURE_OPENAI_GPT_6_SOL_DEPLOYMENT"]
assets = find_assets()

expected_assets = [
    assets / "products.json",
    assets / "manuals/product_info_1.md",
    assets / "manuals/product_info_2.md",
    assets / "images/product_1.webp",
    assets / "images/product_2.webp",
]
missing_assets = [str(path) for path in expected_assets if not path.is_file()]
if missing_assets:
    raise FileNotFoundError(f"Shared Contoso Outdoors files are missing: {missing_assets}")

print(f"Environment ready for deployment: {deployment}")
print(f"Shared assets: {assets}")


## 3. Read before we recommend

We load both product records and manuals. The images keep the scenario tangible,
but this exercise deliberately reasons over written claims only so the success
criterion stays focused.


In [ ]:
# 4. Load the local product evidence
import base64
import json

from IPython.display import HTML, display

# These are the only product records used in this phase.
products = json.loads((assets / "products.json").read_text(encoding="utf-8"))
manuals = {
    product["id"]: (assets / product["manual"]).read_text(encoding="utf-8")
    for product in products
}

def display_webp(path: Path, width: int = 240) -> None:
    """Render a local WebP through HTML because IPython Image cannot embed it."""
    encoded = base64.b64encode(path.read_bytes()).decode("ascii")
    display(HTML(f'<img src="data:image/webp;base64,{encoded}" width="{width}">'))


for product in products:
    print(f'{product["id"]}: {product["name"]} (${product["price"]})')
    display_webp(assets / product["images"][0])

assert [product["id"] for product in products] == [1, 2]


## 5. Test the constraints

The shopper wants shelter for four people and a backpack for a wet hike. They
also assume every advertised accessory is included. The response must identify
what the sources support, where they disagree, and what must be confirmed.


In [ ]:
# 6. Reason over the catalog and manuals
evidence = {
    "catalog": products,
    "manuals": manuals,
}
prompt = f"""
Act as a cautious retail advisor. A customer needs:
- shelter for four people on a rainy three-season campsite;
- a hiking backpack with hydration support;
- included rain protection for both products.

Analyze only the evidence below. Return these headings:
SUPPORTED, CONFLICTS, UNKNOWNS, RECOMMENDATION, SAFETY CHECK.
For every point, cite product ID 1 or 2 and the CATALOG or MANUAL source.
Do not resolve a conflict by guessing.

EVIDENCE
{json.dumps(evidence, indent=2)}
"""

response = client.responses.create(
    model=deployment,
    input=prompt,
    reasoning={"effort": "high"},
)
answer = response.output_text
print(answer)

# Structural checks make omissions visible without pretending to grade the answer.
for heading in ("SUPPORTED", "CONFLICTS", "UNKNOWNS", "RECOMMENDATION", "SAFETY CHECK"):
    assert heading in answer.upper(), f"Missing section: {heading}"
for product in products:
    assert str(product["id"]) in answer, f"Missing product ID {product['id']}"
print(f"\nReasoning usage: {response.usage.output_tokens_details}")


## 7. Your Turn to Explore

- Change the trip to winter and see whether the model declines unsupported use.
- Ask for a claim-by-claim contradiction table.
- Compare `medium` and `high` reasoning effort on the same fixed evidence.


## 8. Summary

We used GPT-6 Sol for one capability: reasoning through constraints and
contradictory evidence. This pattern fits high-value decisions where an
unsupported assumption is more costly than a longer response. Continue with
the [reasoning primer](../../../docs/primers/reasoning-models.md) and the
[context window glossary entry](../../../docs/GLOSSARY.md#context-window).


## 9. References

- [GPT-6 Sol model card](https://ai.azure.com/catalog/models/gpt-6-sol) — model positioning.
- [GPT-6 Astra, Sol, and Luna in Microsoft Foundry](https://azure.microsoft.com/en-us/blog/gpt-6-astra-sol-and-luna-for-production-agents-in-microsoft-foundry/) — family release and deployment guidance.
- [Azure OpenAI reasoning models](https://learn.microsoft.com/en-us/azure/foundry/openai/how-to/reasoning) — reasoning effort and token accounting.
- [Use the Azure OpenAI Responses API](https://learn.microsoft.com/en-us/azure/foundry/openai/how-to/responses) — v1 Python request pattern.
- [Foundry Models sold by Azure](https://learn.microsoft.com/en-us/azure/foundry/foundry-models/concepts/models-sold-directly-by-azure#gpt-56) — verified model version.
